# HAKE-MER — RoBERTa +M3 / +M4 (H3, H4)

**Backbone:** RoBERTa-base · batch 16, LR 5e-5, 4 epochs, seeds 42/123/456.

Three training blocks; **each block downloads its own zip** when finished (save before starting the next block in case Colab disconnects).

1. +M1+M2+M3 (NRC) → `roberta_m3_nrc_campaign.zip`
2. +M1+M2+M3 (SenticNet) → `roberta_m3_senticnet_campaign.zip`
3. +M1+M2+M3+M4 (SenticNet) → `roberta_m4_senticnet_campaign.zip`

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("Runtime → Change runtime type → GPU")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import subprocess
from getpass import getpass
from pathlib import Path

REPO, WORKDIR = "khalef-khalil/marii", Path("/content/marii")
PUBLIC_URL = f"https://github.com/{REPO}.git"
TRAIN_FLAGS = ["--epochs", "4", "--batch-size", "16", "--lr", "5e-5", "--early-stopping-patience", "0"]
SLUG = "roberta_base"

def clone_repo() -> None:
    if WORKDIR.is_dir():
        subprocess.run(["git", "-C", str(WORKDIR), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(WORKDIR), "reset", "--hard", "origin/main"], check=True)
        return
    r = subprocess.run(["git", "clone", "--depth", "1", PUBLIC_URL, str(WORKDIR)], capture_output=True)
    if r.returncode == 0:
        return
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = getpass("GitHub token: ")
    subprocess.run(["git", "clone", "--depth", "1", f"https://{token}@github.com/{REPO}.git", str(WORKDIR)], check=True)

clone_repo()
%cd {WORKDIR}
!git rev-parse --short HEAD

In [ ]:
!pip install -q -r requirements-train.txt
!chmod +x run_m1_m2_m3_campaign.sh run_m1_m2_m3_m4_campaign.sh

In [ ]:
import json
import subprocess
import zipfile
from google.colab import files
from pathlib import Path

def zip_and_download(campaign_json: str, metrics_glob: str, zip_name: str) -> None:
    art = Path("reference/artifacts") / campaign_json
    if not art.is_file():
        raise FileNotFoundError(f"Missing {art} — training may have failed.")
    c = json.loads(art.read_text(encoding="utf-8"))
    m = c["test_aggregate"]["f1_macro"]
    gate = c.get("lexicon_gate_abs", {}).get("mean")
    extra = f" |g|={gate:.4f}" if gate is not None else ""
    print(campaign_json, f"F1-macro test {m['mean']:.4f} ± {m['std']:.4f}{extra}")
    zip_path = Path(f"/content/{zip_name}")
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        zf.write(art, art.name)
        for metrics in sorted(Path("runs").glob(metrics_glob)):
            zf.write(metrics, f"{metrics.parent.name}/{metrics.name}")
    print(f"Download {zip_name} ({zip_path.stat().st_size / 1e3:.1f} KB)")
    files.download(str(zip_path))

def run_campaign(script: str, extra_args: list[str]) -> None:
    cmd = [f"./{script}", *extra_args, *TRAIN_FLAGS]
    print(" ".join(cmd))
    subprocess.run(cmd, check=True)

## Block 1 — +M1+M2+M3 (NRC)

In [ ]:
run_campaign(
    "run_m1_m2_m3_campaign.sh",
    ["--lexicon", "nrc", "--backbone", "roberta-base"],
)
zip_and_download(
    f"m1_m2_m3_nrc_{SLUG}_campaign.json",
    f"{SLUG}_seed*_m1_m2_m3_nrc/metrics.json",
    "roberta_m3_nrc_campaign.zip",
)

## Block 2 — +M1+M2+M3 (SenticNet)

In [ ]:
run_campaign(
    "run_m1_m2_m3_campaign.sh",
    ["--lexicon", "senticnet", "--backbone", "roberta-base"],
)
zip_and_download(
    f"m1_m2_m3_senticnet_{SLUG}_campaign.json",
    f"{SLUG}_seed*_m1_m2_m3_senticnet/metrics.json",
    "roberta_m3_senticnet_campaign.zip",
)

## Block 3 — +M1+M2+M3+M4 (SenticNet)

In [ ]:
run_campaign(
    "run_m1_m2_m3_m4_campaign.sh",
    ["--lexicon", "senticnet", "--backbone", "roberta-base"],
)
zip_and_download(
    f"m1_m2_m3_senticnet_m4_{SLUG}_campaign.json",
    f"{SLUG}_seed*_m1_m2_m3_senticnet_m4/metrics.json",
    "roberta_m4_senticnet_campaign.zip",
)

## Integrity

1. Save executed `.ipynb` → `reference/training_records/step_roberta_ablation/colab/`
2. For **each** downloaded zip: `python3 scripts/integrate_roberta_m3_m4_artifacts.py ~/Downloads/<zip>`
3. `python3 scripts/render_roberta_ablation_tex.py` then `./build.sh build`